# NDVI from a satellite image in 10 lines of Python

**Satellite imagery analysis in Python. From a raw Sentinel-2 scene to an NDVI map in just 10 lines of Python.**

<a href="https://colab.research.google.com/github/tommyscodebase/easy-eo-tutorials/blob/main/01-spectral-indices/00-ndvi-in-10-lines.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://www.youtube.com/playlist?list=PLQDjJNQh9NXU"><img src="https://img.shields.io/badge/YouTube-Watch%20the%20series-FF0000?logo=youtube&logoColor=white" alt="Watch the series on YouTube"></a>
<a href="https://github.com/Tommy-Burns/easy-eo"><img src="https://img.shields.io/badge/GitHub-easy--eo-181717?logo=github&logoColor=white" alt="easy-eo on GitHub"></a>
<a href="https://easy-eo.readthedocs.io"><img src="https://img.shields.io/badge/docs-easy--eo.readthedocs.io-8CA1AF?logo=readthedocs&logoColor=white" alt="Documentation"></a>
<a href="https://pypi.org/project/easy-eo/"><img src="https://img.shields.io/pypi/v/easy-eo.svg" alt="PyPI"></a>

---

This episode is part of the [Easy-EO Tutorial series](https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID).

In the next few minutes you will:

1. **Load a real Sentinel-2 scene** from the hosted sample dataset, with no files to download by hand
2. **Compute NDVI**, and plot it
3. **Save the NDVI to disk**

The dataset is fetched on first load and cached. If you are running this notebook locally on your computer, You can run it offline for subsequent runs after the first run.

### Before you run this
easy-eo must be installed. <a href="https://youtu.be/7neK_fxiFFM"><img src="https://img.shields.io/badge/Watch%20this%20video-FF0000?logo=youtube&logoColor=white" alt="Watch this video"></a> or check the [the installation instructions](https://github.com/tommyscodebase/easy-eo-tutorials#setup) here


📓 All notebooks in the series: [tommyscodebase/easy-eo-tutorials](https://github.com/tommyscodebase/easy-eo-tutorials) · 🐛 Found a bug in the library? [Open an issue](https://github.com/Tommy-Burns/easy-eo/issues)

## **Running in Colab?** 
Uncomment and execute the cell below to install `easy-eo`

In [ ]:
# !pip install easy-eo

## 1. Load the sample scene

In [ ]:
from eeo.datasets import load_sample_dataset
from eeo import load_raster

sd = load_sample_dataset()   # instant if you already prefetched. 1, downloads otherwise
scene = load_raster(sd.sentinel2_cog_stacked, band_names=["blue", "green", "red", "nir"])
scene.describe(stats="approx")

## 2. Get to NIR and Red

Three ways to point Easy-EO at a band: a 1-based index, a name (if you loaded with `band_names=`),
or a separate single-band raster entirely. We named our bands above, so we can just use the strings.

In [ ]:
# We named our bands above, so we can address them by name (used below):
#     scene.ndvi(red="red", nir="nir")
#
# Two equivalent alternatives, for reference:
#     scene.ndvi(red=3, nir=4)                                            # by 1-based index

# -- For separate reads
# nir = load_raster(sd.sentinel2_nir) # or load_raster(the path to your local or COG raster file)
# red = load_raster(sd.sentinel2_red) # or load_raster(the path to your local or COG raster file)
# ndvi = nir.ndvi(red)

# -- You can also import ndvi separately if you dont want to chain the operations
# from eeo.analysis import ndvi

# ndvi_ds = ndvi(red=red, nir=nir)

# -- You can also chain for separate single band files
# load_raster(sd.sentinel2_nir).ndvi(load_raster(sd.sentinel2_red))   # separate single-band files

## 3. Compute NDVI

In [ ]:
ndvi = scene.ndvi(red="red", nir="nir", name="NDVI")
type(ndvi), ndvi.get_shape()

Every index in Easy-EO follows the same contract: output is always `float32`, and a pixel that's
nodata in *either* input band comes out as `NaN` — nothing to handle yourself.

In [ ]:
ndvi.plot_raster(
    cmap="Greens",
    colorbar=True,
    colorbar_label="NDVI values",
)

In [ ]:
# Percentile stretching (2nd-98th) is on by default — turn it off to see raw values
ndvi.plot_raster(
    cmap="Greens",
    colorbar=True,
    colorbar_label="NDVI",
    stretch=False
)

## 5. Save it

In [ ]:
ndvi.save_raster("ndvi.tif", driver="GTiff")
print("Saved. Nothing touched disk until this line ran.")

## 6. The one-liner

Everything above, collapsed into a single chain:

In [ ]:
(
    load_raster(sd.sentinel2_cog_stacked, band_names=["blue", "green", "red", "nir"])
    .ndvi(red="red", nir="nir")
    .plot_raster(
        cmap="Greens",
        colorbar=True,
        colorbar_label="NDVI"
    )
)